# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema defines the dataset's record sets and fields, which we explore using their `@id` references. First, let's list all record sets and their associated fields.

In [ ]:
# List all record sets and fields by their @id
record_sets = []
fields_by_record_set = {}
if hasattr(meta, 'recordSet') and meta.recordSet:
    for rs in meta.recordSet:
        rs_id = getattr(rs, '@id', None) or rs.get('@id') if isinstance(rs, dict) else None
        record_sets.append(rs_id)
        # List fields for each record set
        if hasattr(rs, 'field'):
            fields = rs.field if isinstance(rs.field, list) else [rs.field]
        else:
            fields = []
        field_ids = [getattr(f, '@id', None) if hasattr(f, '@id') else f.get('@id') if isinstance(f, dict) else None for f in fields]
        fields_by_record_set[rs_id] = field_ids

    # Print the overview
    for rs_id in record_sets:
        print(f"Record Set @id: {rs_id}")
        print(f"  Fields: {fields_by_record_set[rs_id]}")
else:
    print("No record sets found in metadata.")

Next, let's display a few records from one of the record sets using its `@id`. If there are multiple record sets, choose the first one.

In [ ]:
# Preview some records from the first available record set
if record_sets:
    record_set_id = record_sets[0]
    print(f"Sample records from record set @id: {record_set_id}")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets available for preview.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set into DataFrames
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show columns for the first record set
if record_sets:
    first_rs = record_sets[0]
    print(f"Columns in DataFrame for record set @id {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No DataFrames created. Check record set availability.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Select a numeric field and group field using their `@id`s (from metadata or DataFrame columns).

In [ ]:
# Identify a numeric field for EDA
if record_sets:
    rs_id = record_sets[0]
    df = dataframes[rs_id]
    # Try finding a numeric field, e.g., a score column
    numeric_fields = [col for col in df.columns if ('score' in col.lower() or 'age' in col.lower()) and pd.api.types.is_numeric_dtype(df[col])]
    numeric_field = numeric_fields[0] if numeric_fields else df.select_dtypes(include='number').columns[0] if not df.empty and len(df.select_dtypes(include='number').columns)>0 else None
    if numeric_field:
        print(f"Using numeric field for EDA: {numeric_field}")
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head())
        # Try grouping by categorical field
        possible_group_fields = [col for col in df.columns if ('gender' in col.lower() or 'education' in col.lower() or 'village' in col.lower())]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using their `@id` and DataFrame column names.

In [ ]:
import matplotlib.pyplot as plt

# Example: Visualize numeric field distribution and relationship with group field
if record_sets and numeric_field:
    df = dataframes[record_sets[0]]
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    # Visualize group comparison if available
    if group_field:
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Average {numeric_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We used `mlcroissant` to load the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya.
- Data fields and record sets were referenced via their `@id` attributes.
- This dataset enables analysis of mental health indicators across demographic attributes. Preliminary exploration shows available numeric scores and grouping by gender or education level.
- Further analysis can help inform public health strategies and deepen understanding of mental health trends in Kilifi County and comparable populations.